* https://www.python-course.eu/python3_decorators.php

* Numba (JIT) & Cython ... to be taken from HPDS 02

* https://cython.readthedocs.io/en/latest/src/tutorial/cython_tutorial.html 
* https://pythonprogramming.net/introduction-and-basics-cython-tutorial/ 
* https://www.infoworld.com/article/3252209/cython-tutorial-how-to-speed-up-python.html
* https://riptutorial.com/cython
* https://blog.paperspace.com/boosting-python-scripts-cython/
* https://github.com/adrn/cython-tutorial
* https://github.com/kwmsmith/scipy2013-cython-tutorial

<center><img alt="" src="images/tau-data_banner.png"/></center>
<center><strong><h1>HPDS-01: Introduction to High Performance Data Science via Python - JIT</h1></strong><br />
<img alt="" src="images/PDS_logo.jpg" />

## (C)Taufik Sutanto
## https://tau-data.id/hpds-01/

# “Simplicity is the soul of efficiency.” – Austin Freeman

<img alt="" src="images/meme_compiler.jpg" />

# Python is Often Criticized as "Slow"

<img alt="" src="images/1_Python_VS_TheRest.png" />

# Numba: Just-In-Time (JIT) Compiler

<img alt="" src="images/numba_logo.png" />

* The most effective method for code optimization is to use profiling to identify process "bottlenecks," which can then be optimized.
* Numba is a module that functions as a "Function Decorator" (to be explained).
* Numba compiles these functions in real-time (JIT).

# Code Profiling

<img alt="" src="images/code_bottleneck.png" />

* Image source: https://scoutapm.com/blog/identifying-bottlenecks-and-optimizing-performance-in-a-python-codebase

# Simple Profiling

In [ ]:
import numpy as np
from time import time
from functools import reduce

N = 10**7
X = np.random.rand(N)

def fs1(X):
    s = 0
    for x in X:
        s += x
    return s/len(X)

def fs2(X):
    return sum(X)/len(X)

def fs3(X):
    return reduce(lambda a,c: a+c, X, 0)/len(X)

def fs4(X):
    return np.mean(X)

In [ ]:
start = time()
mean_val = fs1(X)
end = time()
print("First function result = {}, time required = {}".format(mean_val, end-start))

In [ ]:
start = time()
mean_val = fs2(X)
end = time()
print("Second function result = {}, time required = {}".format(mean_val, end-start))

In [ ]:
start = time()
mean_val = fs3(X)
end = time()
print("Third function result = {}, time required = {}".format(mean_val, end-start))

In [ ]:
start = time()
mean_val = fs4(X)
end = time()
print("4th function result = {}, time required = {}".format(mean_val, end-start))

# Cross-Validating the Results

In [ ]:
%timeit fs4(X)

In [ ]:
%%timeit

fs4(X)

# Function Decorators in Python

In [ ]:
def be_grateful():
    print('Alhamdulillah')

In [ ]:
be_grateful()

In [ ]:
def confirm(func):
    def wrapper():
        while True:
            res = input('Are you sure you want to be grateful? [y/n]')
            if res.lower().strip()=='n':
                return
            elif res.lower().strip()=='y':
                func()
                return
    return wrapper 

In [ ]:
fs = confirm(be_grateful)
fs()

# However, this can be simplified using the "@" function decorator.

In [ ]:
@confirm
def be_grateful():
    print('Alhamdulillah')

In [ ]:
be_grateful()

# Numba as a JIT Function Decorator

In [ ]:
from numba import njit

jit_fs1 = njit(fs1)

In [ ]:
%timeit fs1(X)

In [ ]:
%timeit jit_fs1

# How Numba Works

<img alt="" src="images/cara_kerja_numba.png"/>

* https://towardsdatascience.com/numba-weapon-of-mass-optimization-43cdeb76c7da
* IR: Intermediate Representations
* Bytecode Analysis: Intermediate code that is more abstract than machine code
* LLVM: Low Level Virtual Machine, an infrastructure for developing compilers
* NVVM: An IR compiler based on LLVM, designed to represent GPU kernels

# Case Study: Approximating the value of $\Pi$ Using Hit-or-Miss Monte Carlo

<img alt="" src="images/hit-miss_monteCarlo.gif"/>

* Consider a square with a side length of 1. Inside this square is a ¼ circle with a radius of 1, resulting in an area of $\Pi/4$. Therefore, a point (x,y) inside the square but outside the circle will satisfy the inequality $x^2+y^2>1$.
* Using the Monte Carlo method, we will approximate the value of $\Pi$ based on this system.
* We will generate N uniform random numbers [0,1] for (x,y) and check if each point lies within the circle. The ratio of the number of points inside the circle to the total number of points will approximate the ratio of the circle's area to the square's area. Consequently, the value of $\Pi$ can be approximated with the simple formula:
* 4 * (number of points inside circle) / (Total points)

In [ ]:
def hmMC(N):
    NInside = 0.0
    X = np.random.rand(N)
    Y = np.random.rand(N)
    for x,y in zip(X,Y):
        r = x**2 + y**2
        if (r <= 1):
            NInside += 1      
    return 4.0*NInside/N

In [ ]:
# Testing
for i in range(8):
    print(hmMC(10**i), end=',  ')

In [ ]:
%timeit hmMC(10**7)
# Without JIT - CAUTION, this is VERY SLOW ....

In [ ]:
jit_hmMC = njit(hmMC)

In [ ]:
%timeit jit_hmMC(10**7)
# With JIT

# Example 2: The Mandelbrot (Fractal) Function $Z_{n+1}=Z_n + C$

<img alt="" src="images/Mandelbrot_sequence_new.gif"/>

* https://en.wikipedia.org/wiki/Mandelbrot_set
* https://github.com/lmcintosh/ipython-notebooks/blob/master/tutorials/Tutorials%20-%20Numba%20CUDA%20Python.ipynb

In [ ]:
def create_fractal(min_x, max_x, min_y, max_y, image, iters):
    height = image.shape[0]
    width = image.shape[1]
    pixel_size_x = (max_x - min_x) / width
    pixel_size_y = (max_y - min_y) / height
    
    for x in range(width):
        real = min_x + x * pixel_size_x
        for y in range(height):
            imag = min_y + y * pixel_size_y
            color = mandel(real, imag, iters)
            image[y, x] = color

def mandel(x, y, max_iters):
    """
    Given the real and imaginary parts of a complex number,
    determine if it is a candidate for membership in the Mandelbrot
    set given a fixed number of iterations.
    """
    c = complex(x, y)
    z = 0.0j
    for i in range(max_iters):
        z = z*z + c
        if (z.real*z.real + z.imag*z.imag) >= 4:
            return i
    return max_iters

In [ ]:
%%timeit 
image = np.zeros((1024, 1536), dtype = np.uint8)
create_fractal(-2.0, 1.0, -1.0, 1.0, image, 20) 
# Without JIT

In [ ]:
from pylab import imshow, show

image = np.zeros((1024, 1536), dtype = np.uint8)
create_fractal(-2.0, 1.0, -1.0, 1.0, image, 20) 
imshow(image); show()

In [ ]:
from numba import jit

@jit(nopython=True) # Same effect as njit, but with customizable parameters
def mandel(x, y, max_iters):
    """
    Given the real and imaginary parts of a complex number,
    determine if it is a candidate for membership in the Mandelbrot
    set given a fixed number of iterations.
    """
    c = complex(x, y)
    z = 0.0j
    for i in range(max_iters):
        z = z*z + c
        if (z.real*z.real + z.imag*z.imag) >= 4:
            return i
    return max_iters

@jit(nopython=True) # Same effect as njit, but with customizable parameters
def create_fractal(min_x, max_x, min_y, max_y, image, iters):
    height = image.shape[0]
    width = image.shape[1]
    pixel_size_x = (max_x - min_x) / width
    pixel_size_y = (max_y - min_y) / height
    
    for x in range(width):
        real = min_x + x * pixel_size_x
        for y in range(height):
            imag = min_y + y * pixel_size_y
            color = mandel(real, imag, iters)
            image[y, x] = color

In [ ]:
%%timeit 
image = np.zeros((1024, 1536), dtype = np.uint8)
create_fractal(-2.0, 1.0, -1.0, 1.0, image, 20) 
# With JIT

In [ ]:
image = np.zeros((1024, 1536), dtype = np.uint8)
create_fractal(-2.0, 1.0, -1.0, 1.0, image, 20) 
imshow(image); show()

# Numba Cannot Be Used on Dictionaries

In [ ]:
N = 100
D = {i:i**2 for i in range(N)}

def fs(D):
    return sum(D.values())/len(D)

In [ ]:
fs(D)

In [ ]:
@jit(nopython=True)
def fs(D):
    return sum(D.values())/len(D)

In [ ]:
try:
    fs(D)
except Exception as err_:
    print('Error : ', err_) 

## The error can be avoided by removing the "nopython=True" parameter; however, this will result in no performance gain from the Numba function.

# Parallel Programming with Numba

In [ ]:
N = 5*10**6
a = np.random.rand(N).reshape(N)

def fs(a):
    trace = 0
    for i in range(a.shape[0]):
        trace += np.tanh(a[i])
    return a + trace

In [ ]:
%timeit fs(a)

In [ ]:
from numba import prange

@jit(nopython=True, parallel=True)
def fs(a):
    trace = 0
    for i in prange(a.shape[0]):
        trace += np.tanh(a[i])
    return a + trace

In [ ]:
%timeit fs(a)

In [ ]:
@jit(nopython=True, parallel=True, fastmath=True)
def fs(a):
    trace = 0
    for i in prange(a.shape[0]):
        trace += np.tanh(a[i])
    return a + trace

In [ ]:
%timeit fs(a)

# End of Module

<hr>
<img alt="" src="images/looping_funny.png"/>